# 05 — MLlib Ensemble (RandomForest + GBT)

Train and evaluate a Random Forest and Gradient Boosted Tree ensemble on the
assembled Parquet from notebook 04.  
Target: `ndvi_anomaly_harvest` — seasonal NDVI deviation from vineyard mean.


In [ ]:
import os
os.environ.setdefault('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')
os.environ['PYSPARK_PYTHON']        = '/usr/bin/python3.10'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/usr/bin/python3.10'

PARQUET = '../data/spark/assembled'
MODEL_DIR = '../data/spark/models'
os.makedirs(MODEL_DIR, exist_ok=True)


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName('GrapeExpectations-v2-ML')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '200')
    .getOrCreate())

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)


In [ ]:
from pyspark.sql import functions as F

df = spark.read.parquet(PARQUET)
print(f'Loaded: {df.count():,} rows × {len(df.columns)} cols')

TARGET = 'ndvi_anomaly_harvest'

# Feature groups (adjust if column names differ after pivot)
feature_wks   = list(range(1, 36))
feat_mean_cols = [c for c in df.columns if any(c == f'{w}_ndvi_mean' for w in feature_wks)]

terrain_cols = [c for c in df.columns if any(k in c for k in
    ['elev_','slope_','aspect_','curvature_','profile_','plan_','elev_dev'])]

soil_cols = [c for c in df.columns if any(c.startswith(k) for k in
    ['sandtotal','silttotal','claytotal','awc_','cec7','om_r','ph1to1','ec_r',
     'profile_depth','max_depth','frag','dbovendry','caco3','drain','restriction'])]

feature_cols = feat_mean_cols + terrain_cols + soil_cols
print(f'Features: {len(feature_cols)} total'
      f'  (NDVI early-season: {len(feat_mean_cols)}'
      f'  terrain: {len(terrain_cols)}'
      f'  soil: {len(soil_cols)})')


In [ ]:
# Drop rows with null target or any null feature
df_ml = df.select(feature_cols + [TARGET]).dropna()
print(f'Rows after dropna: {df_ml.count():,}')


In [ ]:
from pyspark.ml.feature import VectorAssembler, Imputer
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Impute any remaining NaNs with column median
imputer = Imputer(inputCols=feature_cols, outputCols=[f'{c}_imp' for c in feature_cols])

assembler = VectorAssembler(
    inputCols=[f'{c}_imp' for c in feature_cols],
    outputCol='features'
)

evaluator = RegressionEvaluator(labelCol=TARGET, predictionCol='prediction', metricName='r2')


In [ ]:
# ── Random Forest ──────────────────────────────────────────────────────────
rf = RandomForestRegressor(
    featuresCol='features',
    labelCol=TARGET,
    numTrees=200,
    maxDepth=10,
    seed=42
)

rf_pipeline = Pipeline(stages=[imputer, assembler, rf])

param_grid_rf = (ParamGridBuilder()
    .addGrid(rf.numTrees,  [100, 200])
    .addGrid(rf.maxDepth,  [8, 12])
    .build())

cv_rf = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=param_grid_rf,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

train, test = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train.count():,}  Test: {test.count():,}')

print('Training RF… (this may take a few minutes)')
rf_model = cv_rf.fit(train)


In [ ]:
rf_preds = rf_model.transform(test)
rf_r2    = evaluator.evaluate(rf_preds)
rf_rmse  = RegressionEvaluator(labelCol=TARGET, predictionCol='prediction',
                               metricName='rmse').evaluate(rf_preds)

print(f'RF  Test R²={rf_r2:.4f}  RMSE={rf_rmse:.5f}')


In [ ]:
# ── Gradient Boosted Trees ────────────────────────────────────────────────
gbt = GBTRegressor(
    featuresCol='features',
    labelCol=TARGET,
    maxIter=100,
    maxDepth=6,
    seed=42
)

gbt_pipeline = Pipeline(stages=[imputer, assembler, gbt])

param_grid_gbt = (ParamGridBuilder()
    .addGrid(gbt.maxIter, [50, 100])
    .addGrid(gbt.maxDepth, [5, 7])
    .build())

cv_gbt = CrossValidator(
    estimator=gbt_pipeline,
    estimatorParamMaps=param_grid_gbt,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

print('Training GBT…')
gbt_model = cv_gbt.fit(train)


In [ ]:
gbt_preds = gbt_model.transform(test)
gbt_r2    = evaluator.evaluate(gbt_preds)
gbt_rmse  = RegressionEvaluator(labelCol=TARGET, predictionCol='prediction',
                                metricName='rmse').evaluate(gbt_preds)

print(f'GBT Test R²={gbt_r2:.4f}  RMSE={gbt_rmse:.5f}')


In [ ]:
# ── Feature importances (from best RF model) ──────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

best_rf = rf_model.bestModel.stages[-1]
importances = best_rf.featureImportances.toArray()

feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False)

# Group by category
cats = {'NDVI early-season': feat_mean_cols,
        'Terrain':           terrain_cols,
        'Soil':              soil_cols}

print('Feature importance by group:')
for cat, cols in cats.items():
    total = feat_imp.loc[feat_imp['feature'].isin(cols), 'importance'].sum()
    print(f'  {cat:22s}: {total*100:.1f}%')

print()
print('Top 15 features:')
print(feat_imp.head(15).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top = feat_imp.head(20)
ax.barh(top['feature'][::-1], top['importance'][::-1])
ax.set_xlabel('Importance')
ax.set_title('Top 20 Feature Importances (RF)')
plt.tight_layout()
plt.savefig('../data/spark/feature_importances.png', dpi=150)
plt.show()


In [ ]:
# ── Save best model ─────────────────────────────────────────────────────────
rf_model.bestModel.write().overwrite().save(os.path.join(MODEL_DIR, 'rf_best'))
gbt_model.bestModel.write().overwrite().save(os.path.join(MODEL_DIR, 'gbt_best'))

print(f'Models saved to {MODEL_DIR}/')
print(f'\nFinal summary:')
print(f'  RF  R²={rf_r2:.4f}  RMSE={rf_rmse:.5f}')
print(f'  GBT R²={gbt_r2:.4f}  RMSE={gbt_rmse:.5f}')
